# Data Labelling Pipeline — `amygda_ops_risk_score`

## What this notebook does

This notebook walks you through the **data labelling pipeline** — the first half of the
Amygda Ops Risk Score system.  It takes a raw maintenance log file, extracts operational
knowledge (systems, subsystems, keywords), and produces a **trained labelling model zip**
that the risk score notebook (`02_risk_score.ipynb`) picks up next.

## Step overview

| Step | Method | Re-runnable? | Locked after |
|------|--------|--------------|------|
| 1 | `configure_labelling_pipeline` | Yes | `downsample` or `extract_keywords` starts |
| 2 | `downsample` | Yes (only if `downsample_required`) | `extract_keywords` starts |
| 3 | `extract_keywords` | Yes | `generate_hierarchy` starts |
| 4 | `generate_hierarchy` | Yes | `update_hierarchy` or `generate_weights` starts |
| 5 | `update_hierarchy` | Yes (n times) | `generate_weights` starts |
| 6 | `generate_weights` | Yes | `run_classification` starts |
| 7 | `update_weights` | Yes (n times) | `run_classification` starts |
| 8 | `run_classification` | **ONE-WAY DOOR** | — |

## Ongoing artifacts — your safety net

When you pass `artifact_dir` to `open_session()`, the SDK automatically saves every
step result as a JSON file in that directory after each successful call:

```
artifacts/labelling/
  configure_labelling_pipeline.json
  downsample.json            ← metadata (row counts, next_step, downsampled_path)
  downsampled.parquet        ← actual sampled rows (only when downsample_required)
  extract_keywords.json      ← includes the keyword list and frequency pool
  generate_hierarchy.json    ← includes the hierarchy rows with confidence ratings
  generate_weights.json      ← includes the generated system/subsystem weights
  update_weights.json        ← includes the manually adjusted weights (if run)
  run_classification.json    ← includes the zip_path
```

These files are your safety net: if your kernel restarts mid-session, load a saved
artifact and carry on — no need to re-run completed steps.


### Every time you open this notebook — run this cell first (imports)

Always run imports before anything else, regardless of which path below you take.

In [ ]:
import json, sys, os


# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

# Check SDK version
import amygda_ops_risk_score
print('amygda_ops_risk_score version:', amygda_ops_risk_score.__version__)

print("SDK imported successfully.")

# How to get your API key

The Amygda API portal is coming soon!

Once available, you will be able to:
- Log in at https://portal.amygda.io (COMING SOON)
- Go to API Keys → Create Key, copy your key, paste it below.

In the meantime, to request an API key contact the Amygda CEO directly:
  faizan@amygdalabs.com

api_key is PERMANENT — it is never wiped when a session ends.
─────────────────────────────────────────────────────────────────────────────

In [ ]:
API_KEY = "paste-your-api-key-here"          # ← paste your key here

In [ ]:
ARTIFACT_DIR = "artifacts/labelling/"         # ← folder for all step JSON artifacts

import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)
with open(f"{ARTIFACT_DIR}api_key.txt", "w") as f:
    f.write(API_KEY)
print(f"api_key saved → {ARTIFACT_DIR}api_key.txt")
print(f"API_KEY  : {API_KEY}")
print(f"ARTIFACT_DIR: {ARTIFACT_DIR}")

# Session scenario guidelines 

### API readiness — run `client.wait_until_ready()` before opening a session

The API runs on a cloud service with scale-to-zero.  On a cold start all ML models
need 60–120 s to load.  `wait_until_ready()` polls until the API is fully ready and
returns immediately if it already is.  **Always call it before `open_session()`.**

---

### Case 1 — Starting a brand new session

Use the setup cells below

> **One session per `api_key` at a time.**  If you call `open_session()` while a valid
> session already exists, the API automatically resumes it and returns the same
> `auth_id` — nothing is overwritten or restarted.

---

In [ ]:
# Connect and wait for the API ────────────────────────────────────
# Run this every time you start a fresh session.
#
# The API runs on a cloud service with scale-to-zero.
# wait_until_ready() polls until all ML models are loaded (cold start: 60-120 s).
# On a warm instance it returns immediately.

client = OpsRiskClient() 
client.wait_until_ready()
print("API is ready.")

In [ ]:
# Open a session ──────────────────────────────────────────────────
# open_session() creates a new session — or auto-resumes the existing one
# if a valid session already exists for this api_key.
# Only one session per api_key is allowed at a time.

config  = SessionConfig(name="my-labelling-run")   # ← give it a meaningful name
session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)

AUTH_ID = session.auth_id
print(f"AUTH_ID  : {AUTH_ID}")
print("Save AUTH_ID — you will need it if your kernel restarts.")
print()
print(json.dumps(session.status(), indent=2))

### Case 2 — Kernel restarted (Python variables are gone, server session is still alive)

Skip Case 1.  Use (the kernel-restart cell) instead.

- Paste your saved `AUTH_ID` and run — `get_session()` makes one status check and
  hands you back a live session object.
- Read the printed step states and continue from the first `NONE` or `FAILED` step.
- Reload any lost in-memory variables from the artifact files in `ARTIFACT_DIR`.

If `get_session()` raises `SessionNotFoundError`, the session TTL has expired.
Go back to Case 1 and open a fresh session.


In [ ]:
# Kernel restarted — reconnect to your existing session ─────────────
# Use this cell (instead of Case 1) when your kernel restarted but the session
# on the server is still alive.
#
# 1. Paste the AUTH_ID that was printed when you ran Cell 2c.
# 2. Set ARTIFACT_DIR to the same path you used in Cell 2b.
# 3. Run this cell. get_session() makes one status check — no re-registration needed.
#
# ─────────────────────────────────────────────────────────────────────────────

AUTH_ID      = "ses-paste-your-auth-id-here"   # ← paste your saved AUTH_ID

client  = OpsRiskClient()
client.wait_until_ready()

session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)

status = session.status()
print(json.dumps(status, indent=2))
print()
print("Step state guide:")
print("  DONE    → completed successfully — skip this step")
print("  RUNNING → step is in-flight on the server.")
print("            Call session.status() again in a moment to check if it finished.")
print("  FAILED  → step failed — safe to re-run")
print("  NONE    → not started — proceed normally")
print()
print("To restore a lost variable from an artifact, e.g.:")
print(f"  with open("{ARTIFACT_DIR}generate_hierarchy.json") as f:")
print(f"      hierarchy_result = json.load(f)")
print()
print("If get_session() raised SessionNotFoundError, the TTL has expired.")
print("open a fresh session (no re-registration needed).")

### Case 3 — Network dropped and came back (kernel still running)

Your in-memory `session` object is still valid — just continue.
The SDK reconnects automatically on the next call.

If a long step was in-flight when the network dropped, call `session.status()` to check
whether it completed on the server, then resume from there.

If `session.status()` raises `APIError (404)`, the session expired during the outage.
Call `client.open_session()` again — the API auto-resumes if the session is still alive,
or creates a fresh one.  **Your `api_key` never changes — it is permanent.**

---

In [ ]:
# session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)
# print(json.dumps(session.status(), indent=2))

### Case 4 — A step is locked and you need to go back to an earlier step

Steps lock once a downstream step runs.  The only way to unlock is to delete and
recreate the session: 

```python
session = client.restart_session(session, api_key=API_KEY, config=config)
# All step locks cleared — re-run from Step 1.
```
This handles deletion and re-creation in one call. 

You can also called session.delete(). Call this directly only if you need to free up the session slot without immediately openinga new one.

⚠ **This permanently deletes all server-side data for the current session.**
Download any artifacts you need before calling this.  Your `api_key` is never affected.

In [ ]:
# session = client.restart_session(session, api_key=API_KEY, config=config)
# print("Session restarted — re-run from Step 1.")

### Note — `next_step` in every response

Every step method returns a dict that includes a `next_step` key telling you what to call next:

```python
result = session.configure_labelling_pipeline(...)
print(result['next_step'])  # e.g. 'extract_keywords'
```

| Step completed | `next_step` value |
|---|---|
| `configure_labelling_pipeline` | `downsample` or `extract_keywords` |
| `downsample` | `extract_keywords` |

| `extract_keywords` | `generate_hierarchy` |
| `generate_hierarchy` | `generate_weights` |
| `update_hierarchy` *(optional)* | `generate_weights` |
| `generate_weights` | `run_classification` |
| `update_weights` *(optional)* | `run_classification` |
| `run_classification` | `configure_training` *(continue in 02_risk_score.ipynb)* |

> **Note:** `next_step` values match both the session status step key and the SDK method name exactly

# Running labelling pipeline

## Step 1 — configure_labelling_pipeline

Upload your log file and lock in the pipeline configuration.  This tells the server
which column contains the log text, what kind of asset you are analysing, and how many
systems and subsystems the LLM should target when building the hierarchy.

**Re-run rule:** Can re-run anytime until Step 2 or 3 starts.

Key fields in the response:
- `row_count` — total rows in the uploaded file
- `downsample_required` — `true` if the dataset exceeds the classification limit
  (in that case, run Step 2 before Step 3)
- `available_columns` — all columns found; useful for double-checking column names

The result is automatically saved to `{ARTIFACT_DIR}configure.json`.

In [ ]:
# ── Step 1 params — edit here ─────────────────────────────────────────────────
FILE_PATH       = "test_files/labelling_model_training_data.csv"   # path to your .csv / .xlsx / .xls file
LOG_COLUMN      = "cleaned_event_details"        # column that contains the log text
ASSET_CONTEXT   = "x-ray baggage handling machine"          # plain-English description of the asset type
MAX_SYSTEMS     = 10                         # how many top-level systems to generate (1–10)
MAX_SUBSYSTEMS  = 5                          # subsystems per system (1–5)
IS_FREE_TEXT    = False                        # True = unstructured free text, False = structured codes
SHEET_NAME      = None                      # XLSX only — None = auto-detect single sheet

configure_result = session.configure_labelling_pipeline(
    file_path=FILE_PATH,
    log_column=LOG_COLUMN,
    max_systems=MAX_SYSTEMS,
    max_subsystems=MAX_SUBSYSTEMS,
    asset_context=ASSET_CONTEXT,
    is_free_text=IS_FREE_TEXT,
    sheet_name=SHEET_NAME,
)
# configure_result is automatically saved to {ARTIFACT_DIR}configure_labelling_pipeline.json
print(json.dumps(configure_result, indent=2))

## Step 2 — downsample  _(skip if `downsample_required` is False)_

If `configure_result['downsample_required']` is `True`, your dataset is too large for
direct classification.  This step applies stratified sampling to bring the row count
within the limit while preserving the distribution across asset types.

**At least one stratification column is required.**  Providing more columns gives
better-balanced samples.

When `artifact_dir` is set, the SDK automatically:
- Saves step metadata to `{ARTIFACT_DIR}downsample.json`
- Downloads the actual sampled rows as `{ARTIFACT_DIR}downsampled.parquet`

The parquet path is returned as `downsampled_path` in the result, and you can inspect
the sampled rows directly after this cell — no manual unzipping required.

**Re-run rule:** Can re-run anytime until Step 3 starts.  
**Locked once:** `extract_keywords` is called.

**Kernel restart recovery:** Load the parquet from the artifact directory:

```python
import pandas as pd
downsampled_df = pd.read_parquet(f"{ARTIFACT_DIR}downsampled.parquet")
```

In [ ]:
# ── Step 2 params — edit here ─────────────────────────────────────────────────
ASSET_COLUMN     = "Asset No"   # unique asset identifier — None if not available
VEHICLE_COLUMN   = None          # vehicle / fleet grouping — None if not available
TIMESTAMP_COLUMN = None          # event timestamp column — None if not available
DOWNSAMPLE_SIZE  = 5000          # target rows after sampling (max 5 000)

if configure_result.get("downsample_required"):
    downsample_result = session.downsample(
        sample_size=DOWNSAMPLE_SIZE,
        asset_column=ASSET_COLUMN,
        vehicle_column=VEHICLE_COLUMN,
        timestamp_column=TIMESTAMP_COLUMN,
        file_path=FILE_PATH,    # validates column names client-side before the API call
        sheet_name=SHEET_NAME,
    )
    # downsample_result is automatically saved to {ARTIFACT_DIR}downsample.json
    # downsampled.parquet is automatically downloaded to {ARTIFACT_DIR}downsampled.parquet
    print(json.dumps(downsample_result, indent=2))

    downsampled_path = downsample_result.get("downsampled_path")
    if downsampled_path:
        print(f"\nDownsampled rows saved to: {downsampled_path}")
        print("Inspect with:")
        print("  import pandas as pd")
        print(f"  pd.read_parquet('{downsampled_path}')")
else:
    print("Downsampling not required — skipped.")

In [ ]:
# ── Inspect the downsampled rows ──────────────────────────────────────────────
# Works immediately after downsample, or after kernel restart from the parquet file.
# Skipped automatically when downsampling was not required.
import os, pandas as pd

if configure_result.get("downsample_required"):
    downsampled_path = (
        downsample_result.get("downsampled_path")         # live result from above
        if "downsample_result" in dir()
        else f"{ARTIFACT_DIR}downsampled.parquet"         # kernel-restart fallback
    )
    if downsampled_path and os.path.exists(downsampled_path):
        downsampled_df = pd.read_parquet(downsampled_path)
        print(f"Sampled rows: {len(downsampled_df):,}  |  Columns: {list(downsampled_df.columns)}")
        display(downsampled_df.head())
    else:
        print(f"Parquet not found at {downsampled_path} — re-run Step 2 to regenerate.")
else:
    print("Downsampling was not required — full dataset is used directly.")

## Step 3 — extract_keywords

Scan the log text and extract a vocabulary of domain-specific terms.  These keywords
become the raw material the LLM uses in Step 4 to infer the operational hierarchy.

**Extraction methods:**
- `"fast"` (default) — TF-IDF-based, quick, recommended for most datasets
- `"deep"` — spaCy NLP, slower but handles noisy or highly abbreviated text better

**`linguistic_filter` parameter (default `True`):**  
When `True`, the extracted n-gram pool is filtered by a dictionary check to remove
n-grams containing non-standard words.  Set to `False` when your domain uses
jargon or abbreviations that the dictionary incorrectly rejects
(e.g. aviation codes such as `APU`, `PTU`, `QRH`).

The response includes:
- `keywords` — the final keyword list sent to the LLM
- `raw_keyword_count` / `filtered_keyword_count` — how many were extracted before / after filtering
- `linguistic_keywords_removed` — n-grams removed by the linguistic filter (`0` when `linguistic_filter=False`)
- `keyword_pool` — `{keyword: frequency}` dict; frequencies reflect how often each term appeared in the logs

**For large datasets (100k+ rows):** increase `timeout` to avoid ReadTimeout.

**Re-run rule:** Can re-run anytime until Step 4 starts.  
**Locked once:** `generate_hierarchy` is called.

**Kernel restart recovery:** If your kernel restarts after this step completes,
reload the saved artifact instead of re-running the step:

```python
with open(f'{ARTIFACT_DIR}extract_keywords.json') as f:
    keywords_result = json.load(f)
keywords = keywords_result['keywords']
```


In [ ]:
# ── Step 3 params — edit here ─────────────────────────────────────────────────
EXTRACTION_METHOD = "deep"   # "fast" (default, recommended) | "deep" (slower, better for noisy text)
LINGUISTIC_FILTER = True     # False = skip dictionary filter; use when domain jargon is being incorrectly removed

keywords_result = session.extract_keywords(
    extraction_method=EXTRACTION_METHOD,
    linguistic_filter=LINGUISTIC_FILTER,
    timeout=1200.0,
)
# keywords_result is automatically saved to {ARTIFACT_DIR}extract_keywords.json

keywords = keywords_result["keywords"]
print(f"Extracted {len(keywords)} keywords")
print(f"Raw count: {keywords_result['raw_keyword_count']}  →  Filtered: {keywords_result['filtered_keyword_count']}")
print(f"Linguistic filter removed: {keywords_result.get('linguistic_keywords_removed', 0)} terms")
print()
print(json.dumps(keywords_result, indent=2))


In [ ]:
# ── Optional: visualise keywords ──────────────────────────────────────────────
# All three forms below work equally:
#   (a) pass the result dict directly
#   (b) pass a file path to the saved artifact  ← useful after kernel restart
#   (c) pass the plain list of strings

# Word cloud (requires: pip install wordcloud matplotlib)
helpers.plot_keyword_cloud(keywords_result, title="Extracted Keywords", max_words=80)

# File path form — works even after kernel restart without re-running the step:
# helpers.plot_keyword_cloud(f"{ARTIFACT_DIR}extract_keywords.json")

In [ ]:
# ── Keyword frequency bar chart ───────────────────────────────────────────────
# Horizontal Plotly bar — highest-frequency terms at the top.
# Bar height = how many times each term appeared in the raw log text
# (from keyword_pool returned by extract_keywords — real counts, not just rank).
#
# Pass the result dict directly — frequencies are read from keyword_pool automatically.
# Also accepts: a file path to the artifact, a plain {keyword: freq} dict, or a list.

helpers.plot_top_keywords(
    keywords_result['keyword_pool'],
    top_n=50,
    title="Top 50 Extracted Keywords by Frequency",
)

# After kernel restart, use the saved artifact (frequencies preserved):
# helpers.plot_top_keywords(f"{ARTIFACT_DIR}extract_keywords.json", top_n=50)

In [ ]:
# ── Keyword frequency table ───────────────────────────────────────────────────
# Returns a DataFrame with columns ["Keyword", "Frequency"], sorted descending.
# Useful for exporting or inspecting exact counts.

df_keywords = helpers.get_top_keywords_table(keywords_result['keyword_pool'], top_n=50)
print(f"Top {len(df_keywords)} keywords:")
display(df_keywords)

# After kernel restart:
# df_keywords = helpers.get_top_keywords_table(f"{ARTIFACT_DIR}extract_keywords.json")

## Step 4 — generate_hierarchy

Send the extracted keywords to the LLM and have it build a two-level hierarchy:
systems (top level) and subsystems (children).  Each row receives a confidence rating
(`high`, `medium`, `low`) that reflects how strongly the keyword evidence supports
that classification.

This step runs synchronously on the server — the SDK blocks until complete
and prints rotating progress messages while you wait.  Typical runtime: 30 s – 3 min.

The response includes:
- `hierarchy` — list of `{system, system_confidence, subsystem, subsystem_confidence}` dicts
- `systems_count` / `subsystems_count` — number of unique systems and subsystems produced
- `keywords_source` — `"extract_keywords"` or `"user_override"`
- `keywords_finalised` — the exact keyword list that reached the LLM after server-side filtering
- `keywords_dropped` — keywords removed by the server's linguistic filter (`null` when using `extract_keywords` defaults)
- `completed_at` — ISO timestamp of when the LLM step finished server-side

**Optional — override keywords:** Pass `keywords=[...]` to use your own keyword list instead
of the ones from `extract_keywords`.  The server applies the same linguistic filtering.
Useful for testing specific vocabularies or trimming the keyword pool manually.

**Re-run rule:** Can re-run anytime until `update_hierarchy` or `generate_weights` starts.

**Kernel restart recovery:** The result is saved to `{ARTIFACT_DIR}generate_hierarchy.json`.
Load it with `helpers.make_hierarchy_rows()` or pass the path directly to any helper.


In [ ]:
hierarchy_result = session.generate_hierarchy(
    timeout=1200,     # increase for large datasets or slow LLM
)
# hierarchy_result is automatically saved to {ARTIFACT_DIR}generate_hierarchy.json

print(f"Systems:    {hierarchy_result['systems_count']}")
print(f"Subsystems: {hierarchy_result['subsystems_count']}")
print(f"Keywords used: {len(hierarchy_result.get('keywords_finalised', []))}  (source: {hierarchy_result.get('keywords_source')})")
dropped = hierarchy_result.get('keywords_dropped')
if dropped:
    print(f"Keywords dropped by server filter: {len(dropped)}")
print()
print(json.dumps(hierarchy_result, indent=2))

# Optional: pass your own keywords to override the extracted ones
# hierarchy_result = session.generate_hierarchy(
#     keywords=["keyword1", "keyword2", ...],
#     timeout=1200,
# )


In [ ]:
# ── Optional: visualise the hierarchy ────────────────────────────────────────
# Pass the result dict, a file path, or a saved artifact path — all work.
# Note:- Colours :- "high":"green", "medium":"yellow", "low":"red"
        
helpers.plot_hierarchy(hierarchy_result, title="System / Subsystem Hierarchy")

# After kernel restart, use the saved artifact:
# helpers.plot_hierarchy(f"{ARTIFACT_DIR}generate_hierarchy.json")

In [ ]:
# ── View the hierarchy as a DataFrame ────────────────────────────────────────
import pandas as pd
hierarchy_df = pd.DataFrame(hierarchy_result["hierarchy"])
hierarchy_df.iloc[:,0:4]

## Step 5 — update_hierarchy  _(optional, repeatable)_

Refine the LLM-generated hierarchy before locking it.  Each call **replaces** the
entire stored hierarchy — pass the complete desired state every time (not a patch).

**What you can change:**

| Operation | How |
|-----------|-----|
| Add a new system or subsystem | Append a new dict to the rows list |
| Delete a system or subsystem | Remove its dict from the list |
| Rename a system or subsystem | Change the `system` / `subsystem` string value |
| Change confidence | Set `system_confidence` / `subsystem_confidence` to `'high'`, `'medium'`, or `'low'` |

**Validation rules (checked client-side before any network call):**
- `system` and `subsystem` must be non-blank strings
- `system_confidence` and `subsystem_confidence` must be `'high'`, `'medium'`, or `'low'` (case-insensitive)
- Each `(system, subsystem)` pair must be unique — duplicates raise `ValidationError`
- **`system_confidence` is a system-level attribute** — all rows for the same system must carry the **same** value. Setting `high` on one row and `low` on another row of the same system raises `ValidationError`. `subsystem_confidence` is per-row and may differ freely across subsystems.
- Only four keys are allowed per row: `system`, `system_confidence`, `subsystem`, `subsystem_confidence`
- Do not pass extra keys (e.g. `keywords`) — any unknown field raises `ValidationError`

**Two ways to edit:**
1. **CSV workflow:** Export to CSV with `helpers.hierarchy_to_csv()`, open in Excel/Numbers, edit and save, then reload with `helpers.hierarchy_from_csv()`
2. **Direct Python:** Edit the `rows` list in-notebook — change dict values, add new dicts, or remove dicts from the list

**Re-run rule:** Can re-run as many times as needed until `generate_weights` starts.

**Kernel restart recovery:**
```python
import json
with open(f'{ARTIFACT_DIR}generate_hierarchy.json') as f:
    prev = json.load(f)
rows = helpers.make_hierarchy_rows(prev)
# ... edit rows ...
update_hierarchy_result = session.update_hierarchy(rows)
```

In [ ]:
# ── Export hierarchy to CSV for external editing ─────────────────────────────
# Saves the hierarchy as a CSV — open in Excel/Numbers, make your changes, save.
# Then load it back in the next cell and submit to update_hierarchy.

HIERARCHY_CSV = f"{ARTIFACT_DIR}hierarchy_draft.csv"
helpers.hierarchy_to_csv(hierarchy_result, HIERARCHY_CSV)
print(f"Hierarchy exported to: {HIERARCHY_CSV}")
print("Edit the file externally, then run the cell below to apply your changes.")

In [ ]:
# ── Load edited hierarchy CSV and submit ─────────────────────────────────────
# Run this after saving your edits to the CSV file above.

rows_from_csv = helpers.hierarchy_from_csv(HIERARCHY_CSV)
update_hierarchy_result = session.update_hierarchy(rows_from_csv)


In [ ]:
# View the updated hierarchy as a DataFrame again to confirm your changes took effect:
updated_hierarchy_result_df = pd.DataFrame(update_hierarchy_result["hierarchy"])
updated_hierarchy_result_df.iloc[:,0:4]

In [ ]:
# Visualise the updated hierarchy
helpers.plot_hierarchy(update_hierarchy_result, title="Updated System / Subsystem Hierarchy")

## Step 6 — generate_weights

Lock the current hierarchy and generate criticality weights for every system and
subsystem.  The weights determine how much each system contributes to the final
operational risk score.

**Which hierarchy does it use?**  
It always uses whatever hierarchy is current in the session:
- If you ran `update_hierarchy` → the latest `update_hierarchy` result is used
- If you only ran `generate_hierarchy` → that result is used  

Both steps write to the same storage location, so the last one wins.  If you ran
`update_hierarchy` multiple times, the most recent run is used.

**Re-run rule:** Can re-run anytime until `run_classification` starts.

**Kernel restart recovery:** Result saved to `{ARTIFACT_DIR}generate_weights.json`.

In [ ]:
weights_result = session.generate_weights()
# weights_result is automatically saved to {ARTIFACT_DIR}generate_weights.json
print(json.dumps(weights_result, indent=2))

In [ ]:
import pandas as pd
# Flat table: one row per subsystem with parent system weight alongside
rows = []
for s in weights_result["weights"]:
    for sub in s["subsystems"]:
        rows.append({
            "system":             s["system_name"],
            "system_weight":      s["weight"],
            "subsystem":          sub["subsystem_name"],
            "subsystem_weight":   sub["weight"],
        })

df_weights = pd.DataFrame(rows)
df_weights

In [ ]:
# ── Optional: visualise the generated weights ─────────────────────────────────
helpers.plot_weights(weights_result, title="System / Subsystem Weights")

# After kernel restart, load from artifact:
# helpers.plot_weights(f"{ARTIFACT_DIR}generate_weights.json")

## Step 7 — update_weights  _(optional, repeatable)_

Override the system and subsystem criticality weights generated in Step 6.
The weights determine how much each system contributes to the aggregate operational
risk score (e.g. a system with weight `0.3` accounts for 30% of the total score).

**How normalisation works (automatic):**
`update_weights` normalises your weights automatically before sending — you do **not**
need to ensure they sum to 1.0 first. It locks the values you changed and proportionally
scales only the untouched ones to fill the remaining budget (same as the UI behaviour).

**Validation rules (run after normalisation):**
- Each individual weight must be between **0.01 and 1.0**. A weight that rounds below 0.01 after normalisation will fail
- Sums must equal 1.0 with a tolerance of **±0.0001** (handles float rounding e.g. `0.33 + 0.33 + 0.34`)

**Name and order lock:**
- The **values** of `system_name` and `subsystem_name` cannot be changed — any rename raises `ValidationError`
- The **order** of systems and of subsystems within each system must stay the same — reordering raises `ValidationError`
- Only the numeric **weight values** may differ from the original
- Always pass `original_systems=weights_result["weights"]` — this is what activates the lock and protects the hierarchy match

> This is different from `update_hierarchy`, which has no locks — you can freely add,
> delete, rename, or reorder systems and subsystems there.

**CSV editing rules:**
- The CSV has one row per subsystem, so `system_name` and `system_weight` repeat across rows
- `system_weight` repeats across every row for a system — set the **same value in all rows** for that system. `weights_from_csv` raises a `ValueError` if the values are inconsistent across rows.
- The **column headers** (`system_name`, `system_weight`, `subsystem_name`, `subsystem_weight`) must not be renamed — `weights_from_csv` parses them by name and raises an error if they are missing
- The **values** in `system_name` / `subsystem_name` cells must not be changed (name lock above)
- Rows must stay in the same order (order lock above)
- Only the values in `system_weight` and `subsystem_weight` columns should be edited

**How to edit:**
Export with `helpers.weights_to_csv()`, open in Excel, edit only the
`system_weight` / `subsystem_weight` columns, reload with `helpers.weights_from_csv()`,
then pass directly to `update_weights(systems, original_systems=weights_result["weights"])`.

**Re-run rule:** Can re-run as many times as needed until `run_classification` starts.

**Kernel restart recovery:**
```python
with open(f'{ARTIFACT_DIR}generate_weights.json') as f:
    weights_result = json.load(f)
systems = helpers.make_weight_update(weights_result)
# edit weight values only — do not rename, reorder, or add/remove entries
result_weights = session.update_weights(systems, original_systems=weights_result['weights'])
```

In [ ]:
# ── Export weights to CSV for external editing ───────────────────────────────
# Saves weights as a flat CSV — one row per subsystem.
# Edit the 'system_weight' and 'subsystem_weight' columns in Excel, save, then load below.
# Do NOT rename system_name or subsystem_name columns.

WEIGHTS_CSV = f"{ARTIFACT_DIR}weights_draft.csv"
helpers.weights_to_csv(weights_result, WEIGHTS_CSV)
print(f"Weights exported to: {WEIGHTS_CSV}")
print("Edit the weight values externally, then run the cell below to apply.")

In [ ]:
# ── Load edited weights CSV and submit ───────────────────────────────────────
# Run after saving your edits to the CSV file above.
# update_weights auto-normalises: values you changed are locked, the rest are scaled.
# No need to manually ensure weights sum to 1.0 before passing.

systems_from_csv = helpers.weights_from_csv(WEIGHTS_CSV)
updated_result_weights = session.update_weights(
    systems_from_csv,
    original_systems=weights_result["weights"],  # locks changed values, protects names
)


In [ ]:
# View the updated weights as a DataFrame again to confirm your changes took effect:
import pandas as pd
# Flat table: one row per subsystem with parent system weight alongside
rows = []
for s in updated_result_weights["weights"]:
    for sub in s["subsystems"]:
        rows.append({
            "system":             s["system_name"],
            "system_weight":      s["weight"],
            "subsystem":          sub["subsystem_name"],
            "subsystem_weight":   sub["weight"],
        })

updated_df_weights = pd.DataFrame(rows)
updated_df_weights

In [ ]:
# Visualise the updated weights
helpers.plot_weights(updated_result_weights, title="Updated System / Subsystem Weights")

## Step 8 — run_classification  🔒 ONE-WAY DOOR

**Cannot be repeated.**

Classify every log entry in the dataset against the accepted hierarchy using the
Qdrant vector store and probabilistic inference.  Then package all labelling artifacts
into a zip file.

The SDK will:
1. Trigger classification on the server (runs synchronously)
2. Poll until complete — prints rotating progress messages
3. **Auto-download** `trained_labelling_model_{auth_id}.zip` into `dest_dir`
4. **Auto-wipe** the session (Redis + DB record + storage)

The return value is a dict:
- `zip_path` — absolute path of the downloaded zip
- `dest_dir` — directory where it was saved
- `auth_id` — session id
- `total_rows` — total number of log rows classified

**Pass the zip to the risk score notebook** (`02_risk_score.ipynb`) via `import_model()`.

The result is automatically saved to `{ARTIFACT_DIR}run_classification.json` so you
can retrieve the zip path later even after a kernel restart.

**Kernel restart mid-download:** The session is only wiped after a successful download.
If the download fails, the session remains alive and you can retry by calling
`run_classification()` again (it will skip re-running classification and go straight
to downloading).


In [ ]:
OUTPUT_DIR = "outputs/"    # directory to save the zip into

classification_result = session.run_classification(OUTPUT_DIR, timeout=3600)
# classification_result is automatically saved to {ARTIFACT_DIR}run_classification.json

zip_path = classification_result["zip_path"]
print(f"Zip downloaded to: {zip_path}")
print("Session wiped automatically.")
print()
print("Next: open 02_risk_score.ipynb and pass this zip to import_model():")
print(f'  ZIP_PATH = "{zip_path}"')

## Classification Results — Distribution Analysis

Mirrors the **Classification Results** tab on the Streamlit Data Labelling page.

The three charts below use `all_predictions_exploded.parquet` from inside the zip.
All helpers also accept a plain DataFrame if you prefer to load it yourself.

```python
# Load directly from the zip after kernel restart
import zipfile, pandas as pd
with zipfile.ZipFile(classification_result["zip_path"]) as zf:
    entry = next(n for n in zf.namelist() if "all_predictions_exploded" in n)
    with zf.open(entry) as f:
        exploded_df = pd.read_parquet(f)
```

In [ ]:
# ── Chart 1: Stacked bar — log counts by system and subsystem ─────────────────
# Mirrors "create_system_subsystem_stacked_chart" in the Streamlit app.
import zipfile, pandas as pd

with zipfile.ZipFile(classification_result["zip_path"]) as zf:
    entry = next(n for n in zf.namelist() if "all_predictions_exploded" in n)
    with zf.open(entry) as f:
        exploded_df = pd.read_parquet(f)




In [ ]:
# Chart 1: Stacked bar — log counts by system and subsystem
helpers.plot_classification_stacked(
    exploded_df,
    title="Log Distribution — Systems & Subsystems",
)

In [ ]:
# ── Chart 2: Bar chart — total log count per system ──────────────────────────
# Uses exploded_df loaded in Chart 1 above.

helpers.plot_classification_by_system(
    exploded_df,
    title="Log Distribution per System",
)

In [ ]:
# ── Chart 3: Horizontal bar — top N subsystems by log count ──────────────────
# Uses exploded_df loaded in Chart 1 above.

helpers.plot_classification_by_subsystem(
    exploded_df,
    top_n=20,
    title="Top 20 Subsystems by Log Count",
)